In [3]:
import torch
import sys
import re
import pandas as pd
from tqdm import tqdm
import os
import json

from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from transformers import AutoModelForCausalLM, AutoTokenizer
# from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document
from llama_index.core.node_parser import SentenceSplitter, SemanticSplitterNodeParser

from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core import StorageContext
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.evaluation import SemanticSimilarityEvaluator
from llama_index.core.ingestion import IngestionPipeline

from huggingface_hub import login

from llama_index.core import PromptTemplate
import warnings, logging
from datasets import Dataset
import importlib.metadata
from huggingface_hub import notebook_login

notebook_login()

from pathlib import Path
import requests
import logging
logging.basicConfig(level=logging.WARNING)
# Force reset the root logger level
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("urllib3").setLevel(logging.WARNING)
logging.getLogger("urllib3.connectionpool").setLevel(logging.WARNING)
from llama_index.llms.huggingface import HuggingFaceLLM
import fitz
import pickle
from helpers.rtr_passage_tagging import DocTagging
from flair.data import Sentence
from flair.nn import Classifier
tagger = Classifier.load('ner-fast')

pytorch_model.bin:   0%|          | 0.00/244M [00:00<?, ?B/s]

2026-04-27 07:52:10,818 SequenceTagger predicts: Dictionary with 20 tags: <unk>, O, S-ORG, S-MISC, B-PER, E-PER, S-LOC, B-ORG, E-ORG, I-PER, S-PER, B-MISC, I-MISC, E-MISC, I-ORG, B-LOC, E-LOC, I-LOC, <START>, <STOP>


In [5]:
state_names=["Alabama", "Alaska", "Arizona", "Arkansas", "California", "Colorado", "Connecticut", "Delaware", "Florida", "Georgia", "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas", "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts", "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana", "Nebraska", "Nevada", "New Hampshire", "New Jersey", "New Mexico", "New York", "North Carolina", "North Dakota", "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island", "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah", "Vermont", "Virginia", "Washington", "West Virginia", "Wisconsin", "Wyoming"]

In [ ]:
with open('nodes/semantic_nodes_news.pkl', 'rb') as f1, open('nodes/semantic_nodes_pdf.pkl', 'rb') as f2:
    news = pickle.load(f1)
    pdf = pickle.load(f2)

In [7]:
print(len(pdf))

5664


In [ ]:
import gc
from flair.data import Sentence
import threading
from concurrent.futures import ThreadPoolExecutor

def save_checkpoint(loaded_pkl, output_path, start, total):
    with open(output_path, "wb") as f:
        pickle.dump(loaded_pkl, f)
    print(f"Checkpoint saved at {start}/{total}")

def normalize_locations(locations):
    mapping = {
        "U.S.": "United States",
        "U.S": "United States",
        "US": "United States",
        "USA": "United States",
        "U.S.A.": "United States",
        "United States of America": "United States",
    }
    normalized = []
    for loc in locations:
        normalized.append(mapping.get(loc.strip(), loc.strip()))
    return list(set(normalized))

def extract_locations_from_sentence(sentence):
    locs = []
    for entity in sentence.get_spans("ner"):
        label = entity.get_label("ner")
        if label.value == "LOC" and label.score >= 0.85:
            locs.append(entity.text)
    locs=normalize_locations(list(set(locs)))
    return locs

def process_pkl_batched(loaded_pkl, style, batch_size):  
    total = len(loaded_pkl)
    output_path = f"nodes/semantic_nodes_{style}_with_locations.pkl"
    checkpoint_thread = None

    with torch.no_grad():
        with ThreadPoolExecutor(max_workers=4) as executor:  
            for start in range(0, total, batch_size):
                end = min(start + batch_size, total)
                batch_nodes = loaded_pkl[start:end]
                sentences = list(executor.map(lambda node: Sentence(node.text), batch_nodes))
                tagger.predict(
                    sentences,
                    mini_batch_size=batch_size, 
                    verbose=False,
                )
                for node, sentence in zip(batch_nodes, sentences):
                    node.metadata["extracted_geographies"] = extract_locations_from_sentence(sentence)
                    sentence.clear_embeddings()
                del sentences
                del batch_nodes
                gc.collect()  
                if start % (batch_size) == 0 and start > 0:
                    if checkpoint_thread:
                        checkpoint_thread.join()
                    checkpoint_thread = threading.Thread(
                        target=save_checkpoint, args=(loaded_pkl, output_path, start, total)
                    )
                    checkpoint_thread.start()

                if start % (batch_size) == 0:
                    print(f"Progress is at Node {start}/{total}")

    if checkpoint_thread:
        checkpoint_thread.join()
    with open(output_path, "wb") as f:
        pickle.dump(loaded_pkl, f)
    print(f"Processed {len(loaded_pkl)} nodes. Saved to {output_path}")
    return loaded_pkl
# for node in pdf:
#     if len(node.text) > 20000:
#         node.text = node.text[:5000]

# processed = process_pkl_batched(pdf, "news", batch_size=64)

for node in pdf:
    if len(node.text) > 20000:
        node.text = node.text[:5000]

processed = process_pkl_batched(pdf, "pdf", batch_size=64)

In [ ]:
with open('nodes/semantic_nodes_pdf_with_locations.pkl', 'rb') as f1, open('nodes/semantic_nodes_pdf.pkl', 'rb') as f2:
    pdf = pickle.load(f1)
print(len(pdf))

## Checking

In [ ]:
with open('nodes/semantic_nodes_news_with_locations.pkl', 'rb') as f1, open('nodes/semantic_nodes_pdf_with_locations.pkl', 'rb') as f2:
    newsloc = pickle.load(f1)
    pdfloc = pickle.load(f2)
with open('nodes/semantic_nodes_news.pkl', 'rb') as f1, open('nodes/semantic_nodes_pdf.pkl', 'rb') as f2:
    news = pickle.load(f1)
    pdf = pickle.load(f2)

In [ ]:
news_data = []
for node in newsloc:
    temp=node.metadata
    temp['Text']=node.text
    news_data.append(temp)

df = pd.DataFrame(news_data)
news_full = []
for node in news:
    temp=node.metadata
    temp['Text']=node.text
    news_full.append(temp)

df_full= pd.DataFrame(news_full)
df_merged = df.copy()
df_merged[["source", "url", "published_date", "folder"]] = df_full[
    ["source", "url", "published_date", "folder"]
]

df_merged

In [ ]:
pdf_data = []
for node in pdfloc:
    temp=node.metadata
    temp['Text']=node.text
    pdf_data.append(temp)

df = pd.DataFrame(pdf_data)
pdf_full = []
for node in pdf:
    temp=node.metadata
    temp['Text']=node.text
    pdf_full.append(temp)

df_full= pd.DataFrame(pdf_full)
df2_merged = df.copy()
df2_merged[["source", "start_page", "end_page", "folder"]] = df_full[
    ["source", "start_page", "end_page", "folder"]
]
df2_merged['source']=df2_merged['source'].apply(lambda x: x.replace("/Users/meenuravi/PhD/CoLM/documents/",""))
df2_merged['extracted_geographies']=df2_merged.apply(lambda x: x['folder'] if len(x['extracted_geographies'])==0 else x['extracted_geographies'], axis=1)
df2_merged['extracted_geographies'] = df2_merged['extracted_geographies'].apply(lambda x: x if isinstance(x, list) else [x])
df2_merged


In [ ]:
df2_merged['extracted_geographies'].value_counts()

In [ ]:
news_bge = pickle.load(open("nodes/semantic_nodes_news_with_bge_embeddings.pkl", "rb"))
news_bge_base = pickle.load(open("nodes/semantic_nodes_news_with_bge_base_embeddings.pkl", "rb"))
pdf_bge = pickle.load(open("nodes/semantic_nodes_pdf_with_bge_embeddings.pkl", "rb"))
pdf_bge_base = pickle.load(open("nodes/semantic_nodes_pdf_with_bge_base_embeddings.pkl", "rb"))


In [ ]:
def make_emb_df(nodes, col):
    return pd.DataFrame([{"Text": node.text.strip(), col: node.embedding} for node in nodes])

def merge_embeddings(df, nodes_list):
    df = df.copy()
    df["_key"] = df["Text"].str.strip()
    for nodes, col in nodes_list:
        emb = make_emb_df(nodes, col)
        df = df.merge(emb, left_on="_key", right_on="Text", how="left", suffixes=("", "_drop"))
        df = df.drop(columns=[c for c in df.columns if c.endswith("_drop")])
    df = df.drop(columns=["_key"])  
    return df

df_merged["embedding_bge"] = [node.embedding for node in news_bge]
df_merged["embedding_bge_base"] = [node.embedding for node in news_bge_base]

df2_merged["embedding_bge"] = [node.embedding for node in pdf_bge]
df2_merged["embedding_bge_base"] = [node.embedding for node in pdf_bge_base]

print(df_merged["embedding_bge"].isna().sum(), "news missing bge")
print(df2_merged["embedding_bge"].isna().sum(), "pdf missing bge")

In [ ]:
df_merged.to_pickle("nodes/news_final.pkl")
df2_merged.to_pickle("nodes/pdf_final.pkl")


In [ ]:
df1 = pd.read_pickle("nodes/news_final.pkl")
df2 = pd.read_pickle("nodes/pdf_final.pkl")

print(df1.columns.tolist())
print(df2.columns.tolist())
df1.head(2)